# nb_ingest_rais_bigquery — Ingestão RAIS via BigQuery (Fabric)

**Fonte:** `basedosdados.br_me_rais.microdados_estabelecimentos` (Base dos Dados)  
**Escopo:** 15 municípios · 3 clusters (Santos, Osasco, Mauá) · SP · 2006–presente  
**Saídas:** `bronze_me_rais_microdados` → `silver_rais`  

> **Pré-requisito:** arquivo de credenciais BigQuery em `/lakehouse/default/Files/bd2024-444413-1084f2b9d765.json`

In [ ]:
# ── Célula 1: Setup e Configuração ──
# Carrega configurações centrais (CLUSTERS e funções utilitárias)
%run ./nb_utils_ibge

# Instala dependências de conexão
%pip install google-cloud-bigquery pyarrow db-dtypes --quiet

from google.cloud import bigquery
import pandas as pd
from pyspark.sql.functions import col, trim, lit

In [ ]:
# ── Célula 2: Inicialização do Cliente BigQuery ──
# Pega os municípios dinamicamente do nb_utils_ibge (evita hardcode)
municipios = get_all_municipios()
municipios_sql = ", ".join([f"'{m}'" for m in municipios])

CREDENTIALS_PATH = "/lakehouse/default/Files/bd2024-444413-1084f2b9d765.json"
client = bigquery.Client.from_service_account_json(CREDENTIALS_PATH)

print(f"[OK] Municípios carregados do Config: {len(municipios)}")
print(f"[OK] Cliente BigQuery inicializado.")

In [ ]:
# ── Célula 3: Camada Bronze — Ingestão do BigQuery ──
# Query idêntica à validada no teste local (9 colunas principais)
query_full = f"""
SELECT
    dados.ano                                    AS ano,
    dados.sigla_uf                               AS sigla_uf,
    dados.id_municipio                           AS id_municipio,
    dados.cnae_2                                 AS cnae_2,
    dados.cnae_2_subclasse                       AS cnae_2_subclasse,
    dados.tamanho_estabelecimento                AS tamanho_estabelecimento,
    SUM(dados.quantidade_vinculos_ativos)        AS quantidade_vinculos_ativos,
    SUM(dados.quantidade_vinculos_clt)           AS quantidade_vinculos_clt,
    SUM(dados.quantidade_vinculos_estatutarios)  AS quantidade_vinculos_estatutarios
FROM `basedosdados.br_me_rais.microdados_estabelecimentos` AS dados
WHERE dados.sigla_uf    = 'SP'
  AND dados.id_municipio IN ({municipios_sql})
  AND dados.ano          >= 2006
GROUP BY
    dados.ano, dados.sigla_uf, dados.id_municipio,
    dados.cnae_2, dados.cnae_2_subclasse, dados.tamanho_estabelecimento
"""

print("[INFO] Executando query BigQuery — pode levar alguns minutos...")
df_pandas = client.query(query_full).to_dataframe()
print(f"[OK] BigQuery retornou {len(df_pandas):,} registros")

# Salva na Bronze oficial via Spark
spark_df = spark.createDataFrame(df_pandas)
save_delta(spark_df, "bronze_me_rais_microdados")

In [ ]:
# ── Célula 4: Camada Silver — Limpeza e Enriquecimento ──
# Mapeia clusters dinamicamente a partir da variável CLUSTERS do nb_utils
cluster_rows = [(code, cluster) for cluster, codes in CLUSTERS.items() for code in codes]
df_cluster = spark.createDataFrame(cluster_rows, ["id_municipio_int", "cluster"])

df_silver = (
    spark.table("bronze_me_rais_microdados")
    .withColumn("id_municipio", col("id_municipio").cast("int"))
    .withColumn("ano",          col("ano").cast("int"))
    .withColumn("cnae_2",       trim(col("cnae_2")))
    .withColumn("cnae_2_subclasse", trim(col("cnae_2_subclasse")))
    # Join para trazer o Cluster (SANTOS, OSASCO, MAUA)
    .join(df_cluster, col("id_municipio") == col("id_municipio_int"), "left")
    .drop("id_municipio_int")
    .filter(col("quantidade_vinculos_ativos") >= 0)
)

save_delta(df_silver, "silver_rais")
print(f"[SUCCESS] Tabela silver_rais atualizada.")
display(df_silver.limit(10))